# 005 Agents And Control Flow

这是 LangChain 学习线的第五份 Notebook。

官方参考：

- https://docs.langchain.com/oss/python/langchain/agents
- https://docs.langchain.com/oss/python/langchain/tools
- https://docs.langchain.com/oss/python/langchain/streaming

学习目标：

1. 理解 LangChain agent 的运行循环
2. 理解 ReAct：Reasoning + Acting
3. 看懂模型节点、工具节点和停止条件
4. 对比本仓库 `HarnessChatAgent._run_loop_stream(...)`
5. 判断 LangChain agent 和自研 Harness 分别适合负责什么

---

## 1. Agent 不是一次模型调用

LangChain 官方文档对 agent 的核心描述是：模型和工具组合起来，反复判断、调用工具、观察结果，直到满足停止条件。

你可以先把它理解成：

```text
while not done:
    model decides next action
    if action is tool_call:
        execute tool
        append tool result
        continue
    else:
        return final answer
```

这和本仓库 Harness 的 Query Loop 很像，只是 LangChain 把很多控制流封装在 agent runtime 里。

In [ ]:
%pip install -U langchain langchain-openai python-dotenv

## 2. 先手写一个最小 ReAct 循环

为了不把 LangChain 当黑盒，我们先用普通 Python 模拟 agent loop。

In [1]:
def demo_weather_tool(location: str) -> str:
    return f"{location}: sunny, 25°C"


def fake_model_decide(messages: list[dict]) -> dict:
    last = messages[-1]["content"]
    if "天气" in last and not any(item.get("role") == "tool" for item in messages):
        return {"action": "tool", "tool_name": "demo_weather_tool", "arguments": {"location": "上海"}}
    return {"action": "answer", "content": "根据工具结果，上海演示天气是晴天，25°C。"}


messages = [{"role": "user", "content": "上海天气如何？"}]
ledger = []

for step_no in range(1, 4):
    plan = fake_model_decide(messages)
    ledger.append({"step": step_no, "plan": plan})
    if plan["action"] == "tool":
        result = demo_weather_tool(**plan["arguments"])
        messages.append({"role": "tool", "name": plan["tool_name"], "content": result})
        continue
    messages.append({"role": "assistant", "content": plan["content"]})
    break

print("messages:")
for item in messages:
    print(item)
print("ledger:", ledger)

Value(False)
messages:
{'role': 'user', 'content': '上海天气如何？'}
{'role': 'tool', 'name': 'demo_weather_tool', 'content': '上海: sunny, 25°C'}
{'role': 'assistant', 'content': '根据工具结果，上海演示天气是晴天，25°C。'}
ledger: [{'step': 1, 'plan': {'action': 'tool', 'tool_name': 'demo_weather_tool', 'arguments': {'location': '上海'}}}, {'step': 2, 'plan': {'action': 'answer', 'content': '根据工具结果，上海演示天气是晴天，25°C。'}}]


## 3. 对照本仓库 Harness Query Loop

本仓库的 `_run_loop_stream(...)` 做的是类似的事情：

```text
phase: planning
  -> 生成 plan
  -> action == delegate_batch: 批量 research
  -> action == delegate: subagent policy + 执行
  -> action == tool: permission + execute
  -> action == answer: 停止 loop
```

区别是：Harness 把以下控制面显式暴露出来：

- ledger
- permission
- approval checkpoint
- subagent synthesis
- SSE event
- resume after approval

## 4. 用 LangChain 创建一个 Agent

现在回到 LangChain。`create_agent` 会创建一个图式 agent runtime。

最小输入仍然是：

- model
- tools
- system_prompt
- messages

In [2]:
import os
from pathlib import Path

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI


def load_project_env() -> Path | None:
    current = Path.cwd().resolve()
    for path in [current, *current.parents]:
        env_path = path / ".env"
        if env_path.exists():
            load_dotenv(env_path, override=False)
            return env_path
    return None


@tool
def get_demo_weather(location: str) -> str:
    """Get deterministic demo weather for a city."""
    return f"{location}: sunny, 25°C"


load_project_env()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-5.4-mini")
OPENAI_BASE_URL = os.getenv("OPENAI_BASE_URL") or None

if not OPENAI_API_KEY:
    agent = None
    print("Skip live agent because OPENAI_API_KEY is missing.")
else:
    model = ChatOpenAI(model=OPENAI_MODEL, api_key=OPENAI_API_KEY, base_url=OPENAI_BASE_URL)
    agent = create_agent(
        model=model,
        tools=[get_demo_weather],
        system_prompt="你是一个中文教学助手。需要演示天气时使用工具。",
    )
    print("Agent ready:", OPENAI_MODEL)

Agent ready: qwq


## 5. 可选：调用 Agent 并观察 messages

LangChain agent 返回的结果中通常包含最终 messages。

如果模型触发工具，你会在 messages 中看到 tool call 和 tool result。

In [3]:
if agent is None:
    print("Skip invoke.")
else:
    result = agent.invoke({
        "messages": [
            {"role": "user", "content": "请查询上海的演示天气"}
        ]
    })
    for idx, msg in enumerate(result.get("messages", []), start=1):
        print("---", idx, type(msg).__name__, getattr(msg, "type", ""))
        print(getattr(msg, "content", msg))

--- 1 HumanMessage human
请查询上海的演示天气
--- 2 AIMessage ai



--- 3 ToolMessage tool
上海: sunny, 25°C
--- 4 AIMessage ai


上海的演示天气是：晴天，25°C。


## 6. Control Flow：LangChain 隐藏了什么

LangChain agent 会帮你处理：

- 模型调用
- 工具调用
- 工具结果回填
- 多轮迭代
- 停止条件

但它默认不会替你完成本仓库 Harness 的全部治理：

- 按风险审批工具
- 输出业务自定义 ledger
- 生成 `approval_required` SSE 事件
- 持久化 approval checkpoint
- 分离 coordinator / subagent 上下文

这些仍然需要应用层设计。

## 7. Static tools 和 Dynamic tools

LangChain 文档把 tools 分成两类：

- static tools：创建 agent 时就传入固定工具列表
- dynamic tools：运行时按状态、权限、用户、阶段过滤或注册工具

这和本仓库很接近：

- `ToolRegistry.list_openai_tools()` 是工具暴露面
- `SkillRegistry.select_skills()` 是技能选择
- `decide_permission()` 是执行前权限裁决

区别是，本仓库目前把权限裁决放在工具执行前，而不是通过 LangChain middleware 过滤工具。

## 8. 什么时候用 LangChain agent，什么时候保留 Harness

适合直接用 LangChain agent 的场景：

- 工具风险低
- 不需要复杂审批
- 不需要自定义 SSE ledger
- 想快速组合模型和工具

适合保留自研 Harness 控制面的场景：

- 工具有写操作或安全风险
- 需要用户审批和恢复
- 需要团队可审计 ledger
- 需要 subagent 上下文隔离
- 需要把错误定位到 planning / tool / synthesis / verification

比较实用的路线是：

```text
LangChain 负责 model/tool/agent 基础抽象
Harness 负责权限、上下文、审批、ledger、恢复
```

## 9. 本讲小结

这一讲记住四点：

1. Agent 是循环，不是一次模型调用。
2. LangChain agent 遵循 ReAct 思路：判断、行动、观察、再判断。
3. LangChain 封装了模型节点和工具节点，但不自动等于业务级 Harness。
4. 真实系统要把 agent loop 和权限、ledger、恢复结合起来。

下一讲建议学习：

- 如何把 LangChain 接入 FastAPI
- 如何在本项目里复用 LangChain agent
- 如何保留 Harness 权限和审批边界